# Schedule on Colab - 3 steps
## Variant A: no Google Drive

Release `L6.3.2.8-RC9.2.2-PRODUCTION-HARDENED-RC6`. Full guide: `PRODUCTION_RUN_GUIDE.md` in the package.

| Step | What you do | What happens |
|---|---|---|
| **1. Setup** | Run it, then choose the package ZIP when the button appears. | Installs the pinned solver and opens the package. |
| **2. Your workbook** | Click the button and choose your filled-in weekly input workbook(s). | Each workbook is checked in seconds: **ACCEPTED** or **REJECTED** with the cell to fix. |
| **3. Run** | Pick `MODE` (QUICK is recommended) and run it. | Safety gate (15-30 min), then the schedule; prints a release verdict per workbook; the results ZIP downloads by itself. |

**What you get, per workbook** (the full results, as before): the after-breaks schedule to publish
(`..._BEST_FINAL_AFTER_BREAKS_SCHEDULE.xlsx`, its *Read Me First* tab shows the status), the
before-breaks review copy, the validated alternatives, and every validation and gate report.

Every published schedule has passed the independent validator with 0 hard failures.
Exit code 0 = approved schedule; anything else = not approved, and the run says why.

**Runtime type:** CPU (a GPU does nothing for this solver); pick High-RAM if offered.

Everything stays in this Colab session: **if it disconnects, the results are lost.** For long runs (DEEP) use the Drive notebook (Variant B).


In [ ]:
#@title 1. Setup { display-mode: "form" }
# Installs the pinned solver, then asks for the package ZIP (skipped if this
# session already has it). ortools 9.15.6755 is the pinned RC9.2.2 solver
# runtime (the runner refuses any other build).
!pip -q install "ortools==9.15.6755" "openpyxl>=3.1" "scipy>=1.11" "ruff==0.15.8" 2>&1 | tail -2
import io, json, multiprocessing, os, pathlib, platform, zipfile
from google.colab import files

extracted = pathlib.Path("/content/rc922")
cands = list(extracted.rglob("SCENARIOS.json")) if extracted.exists() else []
if not cands:
    print("Choose the package ZIP: RC9_2_2_PRODUCTION_PACKAGE.zip "
          "(the RC5 line shipped as RC9_2_2_FIX_VALIDATION_RC5.zip).")
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
        z.extractall(extracted)
    cands = list(extracted.rglob("SCENARIOS.json"))
assert cands, "SCENARIOS.json not found - is that the package ZIP?"
PACKAGE_ROOT = str(cands[0].parent)
ROOT = pathlib.Path(PACKAGE_ROOT)
RESULTS_ROOT = ROOT / "results"
RESULTS_ROOT.mkdir(exist_ok=True)
manifest = json.loads((ROOT / "SCENARIOS.json").read_text())
print("ready:", manifest["package"], "| engine", manifest["engine_release"],
      "| sha256", manifest["engine_sha256"][:16], "...")
print("python", platform.python_version(), "| cpus", multiprocessing.cpu_count())


In [ ]:
#@title 2. Your workbook { display-mode: "form" }
# Click "Choose Files" and pick your filled-in weekly input workbook(s) (.xlsx).
# Each one is checked in seconds with the checks the run starts with, so a bad
# cell is found now, not after the safety gate.
import pathlib, subprocess, sys
from google.colab import files
INPUTS = pathlib.Path("/content/inputs")
INPUTS.mkdir(exist_ok=True)
uploaded = files.upload()
candidates = []
for name, data in uploaded.items():
    if not name.lower().endswith(".xlsx"):
        print(f"{name}: skipped (not an .xlsx workbook)")
        continue
    (INPUTS / name).write_bytes(data)
    candidates.append(INPUTS / name)

WORKBOOKS = []
for path in candidates:
    check = subprocess.run([sys.executable, str(ROOT / "tools" / "check_input_workbook.py"), str(path)],
                           capture_output=True, text=True)
    print(check.stdout)
    if check.returncode == 0:
        WORKBOOKS.append(path)
        print(f"ACCEPTED: {path.name}\n")
    else:
        print(check.stderr[-2000:])
        print(f"REJECTED: {path.name} - fix the items above, then run this step again.\n")
print("ready to run:", [p.name for p in WORKBOOKS] or "nothing yet")


In [ ]:
#@title 3. Run { display-mode: "form" }
MODE = "QUICK"  #@param ["QUICK", "DEEP", "OVERNIGHT", "SMOKE"]
# QUICK = best of 2 x 1-hour seeds (about 1 h on 4+ CPUs, 2 h on 2 CPUs).
# DEEP = best of 4 x 1-hour seeds (about 4 h on 2 CPUs). SMOKE = checks the
# workbook and its hard rules only and builds no schedule (a quick readiness test).
# The 15-30 min safety gate (engine fingerprint, solver version, all offline
# tests) runs first, for each workbook, and refuses to run if anything fails.

#@markdown ---
#@markdown **Advanced - usually leave as is**
SEEDS = 0  #@param {type:"integer"}
# 0 = automatic (measured, evidence/seed_portfolio_ab/RESULT.txt): QUICK 2, DEEP 4, OVERNIGHT 6.
STAGE = "FULL_SCHEDULE"  #@param ["FULL_SCHEDULE", "BEFORE_BREAKS_ONLY"]
LANGUAGE_WORKING_WINDOW = "workbook" #@param ["workbook", "OFF", "MINIMUM_ROWS", "ALL_ROWS", "REQUIRED_LANGUAGE_ONLY"]
COVERAGE_MEASURE = "workbook" #@param ["workbook", "INTERVAL_COUNT", "VOLUME_WEIGHTED"]
# "workbook" = what your Instructions sheet says (recommended). INTERVAL_COUNT:
# interval compliance; VOLUME_WEIGHTED: service level (busy intervals weigh more).
RESUME = True  #@param {type:"boolean"}
OVERWRITE = False  #@param {type:"boolean"}
# RESUME continues an interrupted run from its checkpoints. Tick OVERWRITE to
# replace a workbook that already has results.
TIME_LIMIT = 0  #@param {type:"integer"}
# 0 = the mode's own budget. Change MODE rather than TIME_LIMIT.
SINGLE_LONG_RUN = False  #@param {type:"boolean"}
# Keep False: best-of-seeds scored higher than one long run in the measurements.
RUN_PACKAGED_SCENARIOS = False  #@param {type:"boolean"}
ONLY = ""  #@param {type:"string"}
SHARD = 0  #@param {type:"integer"}
SHARDS = 1  #@param {type:"integer"}
# Tick RUN_PACKAGED_SCENARIOS to run the packaged scenarios instead of your
# workbooks; ONLY / SHARD / SHARDS pick among them.
SKIP_GUARDS = False  #@param {type:"boolean"}
# Keep False: the guards are what make the output safe to use.

import multiprocessing, os, signal, subprocess, sys


def run_one(extra):
    """One runner call with today's command and safeguards; returns its exit code."""
    cmd = [sys.executable, "-u", str(ROOT / "runners" / "rc922_runner.py"),
           "--package-root", str(ROOT),
           "--results-root", str(RESULTS_ROOT),
           "--shard", str(SHARD), "--shards", str(SHARDS),
           "--mode", MODE,
           "--stage", STAGE,
           "--num-workers", str(multiprocessing.cpu_count())]
    AUTO_SEEDS = {"SMOKE": 1, "QUICK": 2, "DEEP": 0, "OVERNIGHT": 0}   # 0: runner default (DEEP 4, OVERNIGHT 6)
    seeds = int(SEEDS) if int(SEEDS) > 0 else AUTO_SEEDS[MODE]
    if seeds:                            cmd += ["--seeds", str(seeds)]
    if SINGLE_LONG_RUN:                  cmd += ["--single-run"]
    if TIME_LIMIT:                       cmd += ["--time-limit", str(TIME_LIMIT)]
    if LANGUAGE_WORKING_WINDOW != "workbook":
        cmd += ["--language-working-window", LANGUAGE_WORKING_WINDOW]
    if COVERAGE_MEASURE != "workbook":
        cmd += ["--coverage-objective-weighting", COVERAGE_MEASURE]
    if SKIP_GUARDS:                      cmd += ["--skip-guards"]
    if RESUME:                           cmd += ["--resume"]
    if OVERWRITE:                        cmd += ["--overwrite"]
    cmd += extra
    print(" ".join(cmd), "\n")
    # The runner and everything it starts (seed runs, engines) live in their own
    # process group, so interrupting this cell stops all of them, not just the top one.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, start_new_session=(os.name == "posix"))
    try:
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    except KeyboardInterrupt:
        print("\nInterrupt received: asking the runner to stop safely and preserve checkpoints...")
        try:
            proc.send_signal(signal.SIGINT)
            print("runner exit code:", proc.wait(timeout=30))
        except subprocess.TimeoutExpired:
            if os.name == "posix":
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            else:
                proc.terminate()
            proc.wait(timeout=30)
        raise


if RUN_PACKAGED_SCENARIOS:
    jobs = [("packaged scenarios", ["--only", ONLY.strip()] if ONLY.strip() else [])]
else:
    assert WORKBOOKS, "No accepted workbook yet: run step 2 and upload your workbook first."
    jobs = [(path.name, ["--input", str(path)]) for path in WORKBOOKS]

exit_codes = {}
for label, extra in jobs:
    print(f"\n========== {label} ==========")
    exit_codes[label] = run_one(extra)
    print(f"\n{label}: exit code {exit_codes[label]} (0 = approved schedule; anything else = not approved)")

# Score the release gates for everything in the results folder.
gate = subprocess.run([sys.executable, str(ROOT / "tools" / "release_gate_report.py"),
                       str(RESULTS_ROOT), "--out-dir", str(RESULTS_ROOT / "_gate_report")],
                      capture_output=True, text=True)
print(gate.stdout)
if gate.returncode not in (0, 3):
    print(gate.stderr[-3000:])
print("\nRELEASE VERDICTS")
for line in gate.stdout.splitlines():
    if line.startswith("RELEASE VERDICT"):
        print("  " + line)
print("\nOpen <workbook>/..._BEST_FINAL_AFTER_BREAKS_SCHEDULE.xlsx in each folder: it is the schedule "
      "to publish (Read Me First tab shows its status). The before-breaks file is for review only.")

# Package everything (as before) and download it.
import shutil, time
from google.colab import files
stamp = time.strftime("%Y%m%d_%H%M%S")
archive = shutil.make_archive(f"/content/RC922_RESULTS_{stamp}", "zip", str(RESULTS_ROOT))
print("results:", archive, round(os.path.getsize(archive) / 1e6, 1), "MB")
files.download(archive)
